## Preprocessing related to paper

### Data Ingestion

Copy all the relevant files from the /ETL/data folder after running the ETL pipeline.

These correspond to the following folders:
- annual_electricity_demand
- electricity_demand
- gdp
- temperature

Note: Skip parts of the code for variables you do not want to include in the final dataset.

#### Imports

In [1]:
import os

import pandas
import xarray
from tqdm import tqdm

#### Annual Electricity Demand

In [2]:
# Specify the folder and file type
electricity_annual_demand_folder = "./data/annual_electricity_demand/"
electricity_annual_demand_files = [
    file_name
    for file_name in os.listdir(electricity_annual_demand_folder)
    if file_name.endswith(".parquet")
]

In [3]:
# Load all the files into one DataFrame
df_annual_demand = pandas.DataFrame()

for file_name in tqdm(electricity_annual_demand_files):
    df_current = pandas.read_parquet(
        electricity_annual_demand_folder + file_name
    )

    df_current = df_current.resample(
        "1h", label="right", closed="right"
    ).mean()

    # Add a column for the region name
    df_current["region_code"] = file_name.split(".")[0]

    # Reset index to move "Time (UTC)" to a column
    df_current = df_current.reset_index()

    df_annual_demand = pandas.concat(
        [df_annual_demand, df_current], ignore_index=True
    )

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 97/97 [00:07<00:00, 13.20it/s]


In [4]:
print(df_annual_demand.shape)
df_annual_demand.head()

(9433298, 4)


,Time (UTC),Annual electricity demand (TWh),Annual electricity demand per capita (MWh),region_code
0,2019-01-01 05:00:00,608.95,16.12,CA_QC
1,2019-01-01 06:00:00,608.95,16.12,CA_QC
2,2019-01-01 07:00:00,608.95,16.12,CA_QC
3,2019-01-01 08:00:00,608.95,16.12,CA_QC
4,2019-01-01 09:00:00,608.95,16.12,CA_QC


#### Electricity Demand

In [43]:
# Specify the folder and file type
electricity_demand_folder = "./data/electricity_demand/"
demand_files = [
    file_name
    for file_name in os.listdir(electricity_demand_folder)
    if file_name.endswith(".parquet")
]

In [44]:
# Load all the files into one DataFrame
df_demand = pandas.DataFrame()

for file_name in tqdm(demand_files):
    
    df_current = pandas.read_parquet(electricity_demand_folder + file_name)

    df_current["Load (MW)"] = df_current["Load (MW)"].astype(float)

    df_current = df_current.resample(
        "1h", label="right", closed="right"
    ).mean()

    # Add a column for the region name
    df_current["region_code"] = str.join("_", file_name.split("_")[:-1])

    # Reset index to move "Time (UTC)" to a column
    df_current = df_current.reset_index()

    df_demand = pandas.concat([df_demand, df_current], ignore_index=True)

  0%|                                                                                                                                                           | 0/100 [00:00<?, ?it/s]

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:11<00:00,  8.63it/s]


In [45]:
print(df_demand.shape)
df_demand.head()

(10018507, 3)


,Time (UTC),Load (MW),region_code
0,2019-01-01 06:00:00,23762.55,CA_QC
1,2019-01-01 07:00:00,23830.23,CA_QC
2,2019-01-01 08:00:00,23608.07,CA_QC
3,2019-01-01 09:00:00,23562.48,CA_QC
4,2019-01-01 10:00:00,23546.16,CA_QC


#### GDP

In [8]:
# Specify the folder and file type
gdp_folder = "./data/gdp/"
gdp_files = [
    file_name
    for file_name in os.listdir(gdp_folder)
    if file_name.endswith(".nc")
]

In [9]:
# Load all the files into one DataFrame
df_gdp_data = pandas.DataFrame()

for file_name in tqdm(gdp_files):
    # Extract region code from filename (assuming format like "US_0.25_deg_2020.nc")
    region_code = file_name.split("_0.25_deg_")[0]
    year = int(file_name.split("_0.25_deg_")[-1].replace(".nc", ""))
    
    # Open the NetCDF file
    gdp_data = xarray.open_dataset(gdp_folder + file_name)
    
    # Extract GDP value - assuming the GDP data is stored in a variable named 'gdp'
    # You may need to adjust this based on the actual variable name in your .nc files
    gdp_value = float(gdp_data.gdp.values.sum())
    
    # Create a DataFrame for this file
    df_current = pandas.DataFrame({
        "year": [year],
        "GDP": [gdp_value],
        "region_code": [region_code]
    })
    
    # Extract country code (assuming it's the first part of region_code)
    country_code = region_code.split("_")[0]
    df_current["country_code"] = country_code
    
    # Append to the main DataFrame
    df_gdp_data = pandas.concat([df_gdp_data, df_current], ignore_index=True)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2037/2037 [00:19<00:00, 102.23it/s]


In [10]:
print(df_gdp_data.shape)
df_gdp_data.head()

(2037, 4)


,year,GDP,region_code,country_code
0,2003,3.365960e+12,US_NY,US
1,2014,1.953330e+12,ES,ES
2,2000,1.432630e+10,MX_BCS,MX
3,2019,7.737295e+10,ME,ME
4,2005,6.058992e+12,FR,FR


#### Temperature (Weather)

In [40]:
# Specify the folder and file type
temperature_folder = "./data/temperature/"
temperature_files = [
    file_name
    for file_name in os.listdir(temperature_folder)
    if file_name.endswith(".parquet")
]

In [41]:
# Load all the files into one DataFrame
df_all_temperature = pandas.DataFrame()

for file_name in tqdm(temperature_files):
    df_current = pandas.read_parquet(temperature_folder + file_name)

    # Add a column for the region name
    df_current["region_code"] = file_name.split("_temp")[0]

    # Reset index to move "Time (UTC)" to a column
    df_current = df_current.reset_index()

    df_all_temperature = pandas.concat(
        [df_all_temperature, df_current], ignore_index=True
    )

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1084/1084 [02:27<00:00,  7.37it/s]


In [42]:

print(df_all_temperature.shape)
df_all_temperature.head()

(9433073, 13)


,Time (UTC),Local hour of the day,Local weekend indicator,Local month of the year,Local year,Temperature - Top 1 (K),Temperature - Top 3 (K),Monthly average temperature - Top 1 (K),Monthly average temperature rank - Top 1,Annual average temperature - Top 1 (K),5 percentile temperature - Top 1 (K),95 percentile temperature - Top 1 (K),region_code
0,2024-01-01 08:00:00,0,0,1,2024,285.152832,283.448395,286.329193,11.0,289.368805,284.157623,295.853897,MX_BCA
1,2024-01-01 09:00:00,1,0,1,2024,284.829193,282.712860,286.329193,11.0,289.368805,284.157623,295.853897,MX_BCA
2,2024-01-01 10:00:00,2,0,1,2024,284.859619,281.656281,286.329193,11.0,289.368805,284.157623,295.853897,MX_BCA
3,2024-01-01 11:00:00,3,0,1,2024,284.644073,281.293365,286.329193,11.0,289.368805,284.157623,295.853897,MX_BCA
4,2024-01-01 12:00:00,4,0,1,2024,284.638550,281.604065,286.329193,11.0,289.368805,284.157623,295.853897,MX_BCA


### Merge data into one larger dataset

Keep in mind to skip any code blocks corresponding to variables not loaded in the above code.

The order of merging is based on empirical evidence based on the feature importance after training a XGBoost model.
Feel free to adjust it to your needs.

#### Merge Temperature and Demand Data

In [46]:
df_all_temperature = df_all_temperature.sort_values(by=["Time (UTC)"])
df_demand = df_demand.sort_values(by=["Time (UTC)"])

In [47]:
# Merge the demand data
total_dataset = pandas.merge(
    df_all_temperature, df_demand, on=["Time (UTC)", "region_code"]
)

In [48]:
print(total_dataset.shape)
total_dataset.head()

(8283174, 14)


,Time (UTC),Local hour of the day,Local weekend indicator,Local month of the year,Local year,Temperature - Top 1 (K),Temperature - Top 3 (K),Monthly average temperature - Top 1 (K),Monthly average temperature rank - Top 1,Annual average temperature - Top 1 (K),5 percentile temperature - Top 1 (K),95 percentile temperature - Top 1 (K),region_code,Load (MW)
0,2000-01-01 03:00:00,1,1,1,2000,298.412811,298.300842,298.983521,8.0,299.380615,296.219357,304.797337,BR_N,2373.7
1,2000-01-01 03:00:00,1,1,1,2000,293.334686,293.494202,295.275848,1.0,292.583405,284.913759,300.310954,BR_SE,21183.0
2,2000-01-01 03:00:00,1,1,1,2000,299.080780,298.188843,299.841431,6.0,299.724396,297.591458,302.318542,BR_NE,5340.2
3,2000-01-01 03:00:00,1,1,1,2000,292.053436,293.098358,293.658997,1.0,290.223969,281.118460,298.741829,BR_S,5777.0
4,2000-01-01 04:00:00,2,1,1,2000,298.599792,298.074615,298.983521,8.0,299.380615,296.219357,304.797337,BR_N,2331.6


#### Merge Annual Electricity Demand

In [ ]:
# Run to add annual demand data to the dataset
total_dataset = pandas.merge(
    total_dataset, df_annual_demand, on=["Time (UTC)", "region_code"]
)
# Scale the yearly demand from TW to MW
total_dataset["year_electricity_demand_mw"] = (
    total_dataset["Annual electricity demand (TWh)"] * 1000000
)
total_dataset = total_dataset.drop(columns=["Annual electricity demand (TWh)"])

In [34]:
print(total_dataset.shape)
total_dataset.head()

(8265600, 17)


,Time (UTC),Local hour of the day,Local weekend indicator,Local month of the year,Local year,Temperature - Top 1 (K),Temperature - Top 3 (K),Monthly average temperature - Top 1 (K),Monthly average temperature rank - Top 1,Annual average temperature - Top 1 (K),5 percentile temperature - Top 1 (K),95 percentile temperature - Top 1 (K),region_code,Load (MW),Annual electricity demand (TWh),Annual electricity demand per capita (MWh),year_electricity_demand_mw
0,2000-01-01 03:00:00,1,1,1,2000,298.412811,298.300842,298.983521,8.0,299.380615,296.219357,304.797337,BR_N,2373.7,391.92,2.25,391920000.0
1,2000-01-01 03:00:00,1,1,1,2000,293.334686,293.494202,295.275848,1.0,292.583405,284.913759,300.310954,BR_SE,21183.0,391.92,2.25,391920000.0
2,2000-01-01 03:00:00,1,1,1,2000,299.080780,298.188843,299.841431,6.0,299.724396,297.591458,302.318542,BR_NE,5340.2,391.92,2.25,391920000.0
3,2000-01-01 03:00:00,1,1,1,2000,292.053436,293.098358,293.658997,1.0,290.223969,281.118460,298.741829,BR_S,5777.0,391.92,2.25,391920000.0
4,2000-01-01 04:00:00,2,1,1,2000,298.599792,298.074615,298.983521,8.0,299.380615,296.219357,304.797337,BR_N,2331.6,391.92,2.25,391920000.0


#### Merge GDP

In [ ]:
total_dataset = pandas.merge(
    total_dataset,
    df_gdp_data.drop(columns=["country_code"]),
    left_on=["Local year", "region_code"],
    right_on=["year", "region_code"],
)
total_dataset = total_dataset.drop(columns=["year"])


In [ ]:
print(total_dataset.shape)
total_dataset.head()

#### Renaming to simplify column names

In [49]:
total_dataset = total_dataset.rename(
    columns={
        "Time (UTC)": "time_utc",
        "Local hour of the day": "local_hour",
        "Local weekend indicator": "is_weekend",
        "Local month of the year": "local_month",
        "Local year": "local_year",
        "Temperature - Top 1 (K)": "year_temp_top1",
        "Temperature - Top 3 (K)": "year_temp_top3",
        "Monthly average temperature - Top 1 (K)": "monthly_temp_avg_top1",
        "Monthly average temperature rank - Top 1": "monthly_temp_avg_rank_top1",
        "Annual average temperature - Top 1 (K)": "year_temp_avg_top1",
        "5 percentile temperature - Top 1 (K)": "year_temp_percentile_5",
        "95 percentile temperature - Top 1 (K)": "year_temp_percentile_95",
        "Annual electricity demand (TWh)": "year_electricity_demand",
        "Annual electricity demand per capita (MWh)": "year_electricity_demand_per_capita_mwh",
        "Load (MW)": "load_mw",
        "GDP": "year_gdp",
    }
)
print(total_dataset.shape)
total_dataset.head()

(8283174, 14)


,time_utc,local_hour,is_weekend,local_month,local_year,year_temp_top1,year_temp_top3,monthly_temp_avg_top1,monthly_temp_avg_rank_top1,year_temp_avg_top1,year_temp_percentile_5,year_temp_percentile_95,region_code,load_mw
0,2000-01-01 03:00:00,1,1,1,2000,298.412811,298.300842,298.983521,8.0,299.380615,296.219357,304.797337,BR_N,2373.7
1,2000-01-01 03:00:00,1,1,1,2000,293.334686,293.494202,295.275848,1.0,292.583405,284.913759,300.310954,BR_SE,21183.0
2,2000-01-01 03:00:00,1,1,1,2000,299.080780,298.188843,299.841431,6.0,299.724396,297.591458,302.318542,BR_NE,5340.2
3,2000-01-01 03:00:00,1,1,1,2000,292.053436,293.098358,293.658997,1.0,290.223969,281.118460,298.741829,BR_S,5777.0
4,2000-01-01 04:00:00,2,1,1,2000,298.599792,298.074615,298.983521,8.0,299.380615,296.219357,304.797337,BR_N,2331.6


#### Post-processing

##### Duplicate and NaN removal

In [50]:
# Remove duplicates
row_count = len(total_dataset)
print("Before removing duplicates:", row_count)
total_dataset = total_dataset.drop_duplicates(
    subset=[col for col in total_dataset.columns if col != "load_mw"]
)
print("Without duplicates: ", len(total_dataset))
print("Difference", row_count - len(total_dataset))

Before removing duplicates: 8283174
Without duplicates:  7551648
Difference 731526


In [51]:
# Remove NaN values
row_count = len(total_dataset)
print("Before removing NaN values:", row_count)
total_dataset = total_dataset.dropna()
print("Without duplicates: ", len(total_dataset))
print("Difference", row_count - len(total_dataset))

Before removing NaN values: 7551648
Without duplicates:  7435129
Difference 116519


##### Calculate the percentage that each hour represents of the yearly load

In [53]:
for name, group in total_dataset.groupby(["region_code", "local_year"]):

    yearly_load = group["load_mw"].sum()
    amount_of_hours_tracked = (len(group["load_mw"]))
    # Calculate the amount of hours in the specified year, accounting for leap years
    amount_of_hours_in_year = len(pandas.date_range(start=f"{name[1]}-01-01", end=f"{name[1]}-12-31"))*24

    # Calculate the percentage that load_mw represents of the yearly load
    load_mw_percentage = group["load_mw"] / yearly_load

    # Adjust the percentages to account for missing hours
    total_dataset.loc[group.index, "load_mw_percentage"] = load_mw_percentage * (amount_of_hours_tracked / amount_of_hours_in_year)


### Save the dataset

In [54]:
total_dataset.to_parquet(
    "./data/total_dataset.parquet", engine="pyarrow"
)

#### Details and visuals of the total dataset

In [ ]:
# Investigate the distribution of available hours per region and year
list_amount_hours_region = []
for name, group in total_dataset.groupby(["region_code", "local_year"]):
    list_amount_hours_region.append([name[0], name[1], len(group)])

df_amount_hours_region = pandas.DataFrame(
    list_amount_hours_region,
    columns=["region_code", "local_year", "count_available_hours"],
)

df_amount_hours_region["count_available_hours"].hist(bins=10)